# Bedrock AgentCore MCP Agent

This notebook demonstrates how to build an AI agent on **Amazon Bedrock AgentCore Runtime** that uses the **Model Context Protocol (MCP)** to query a Neo4j graph database containing aviation fleet data.

You'll learn how to:
- Authenticate with an AgentCore Gateway using OAuth2
- Connect to a Neo4j MCP server via `MultiServerMCPClient`
- Build a ReAct agent that generates and executes Cypher queries
- Run natural language queries against aviation fleet data

**Architecture:** `User Question → ReAct Agent (Claude on Bedrock) → MCP Tools → Neo4j Database`

**Aviation Fleet Data:**
The Neo4j database contains Aircraft, Flights, Airports, Operators, Systems, Components, Sensors, Readings, MaintenanceEvents, and Delays — a full aviation fleet operations and maintenance graph.

**Prerequisites:**
- `../CONFIG.txt` configured with MODEL_ID and REGION
- `.mcp-credentials.json` in this directory with AgentCore Gateway credentials

In [ ]:
# Load configuration from CONFIG.txt
from dotenv import load_dotenv
import os

load_dotenv("../CONFIG.txt")

MODEL_ID = os.getenv("MODEL_ID")
REGION = os.getenv("REGION", "us-east-1")

# Derive BASE_MODEL_ID for Claude inference profiles
# us.anthropic.claude-* or global.anthropic.claude-* -> anthropic.claude-*
if MODEL_ID and MODEL_ID.startswith("us.anthropic."):
    BASE_MODEL_ID = MODEL_ID.replace("us.anthropic.", "anthropic.")
elif MODEL_ID and MODEL_ID.startswith("global.anthropic."):
    BASE_MODEL_ID = MODEL_ID.replace("global.anthropic.", "anthropic.")
else:
    BASE_MODEL_ID = None

print(f"Model:  {MODEL_ID}")
if BASE_MODEL_ID:
    print(f"Base:   {BASE_MODEL_ID}")
print(f"Region: {REGION}")

In [ ]:
import importlib.metadata

packages = [
    "langchain",
    "langchain-core",
    "langgraph",
    "langchain-aws",
    "langchain-mcp-adapters",
    "mcp",
    "httpx",
    "boto3",
    "nest-asyncio",
]

print("Pre-installed packages:")
print("-" * 50)
for pkg in packages:
    try:
        version = importlib.metadata.version(pkg)
        print(f"{pkg:30} {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{pkg:30} NOT INSTALLED")

In [ ]:
# Install missing packages
%pip install langgraph>=1.0.6 langchain-mcp-adapters>=0.2.0 mcp>=1.3.0 httpx>=0.28.0 nest-asyncio -q

## 2. Imports

Import the required libraries:
- **LangChain AWS**: `ChatBedrockConverse` for Bedrock integration
- **LangGraph**: `create_react_agent` for building the ReAct agent
- **LangChain MCP Adapters**: `MultiServerMCPClient` bridges MCP tools into LangChain
- **httpx**: For OAuth2 token refresh
- **nest_asyncio**: Enables async MCP calls within Jupyter's event loop

In [ ]:
import json
from datetime import datetime, timedelta, timezone
from pathlib import Path

import httpx
import nest_asyncio
from langchain_aws import ChatBedrockConverse
from langchain_core.messages import HumanMessage
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent

# Allow nested async calls in Jupyter
nest_asyncio.apply()

print("Imports successful!")

## 3. Load MCP Gateway Credentials

The Neo4j MCP server is accessed through an **AgentCore Gateway** endpoint. Authentication uses the OAuth2 client credentials flow via Amazon Cognito.

The `.mcp-credentials.json` file contains:
- `gateway_url` — AgentCore Gateway endpoint for the MCP server
- `token_url` — Cognito OAuth2 token endpoint
- `client_id` / `client_secret` — OAuth2 client credentials
- `access_token` — Current bearer token (auto-refreshed if expired)

In [ ]:
def load_mcp_credentials(filepath=None):
    """Load MCP Gateway credentials from JSON file."""
    if filepath is None:
        filepath = Path(".mcp-credentials.json")
    else:
        filepath = Path(filepath)

    if not filepath.exists():
        raise FileNotFoundError(
            f"Credentials file not found: {filepath}\n"
            "Create .mcp-credentials.json with gateway_url, access_token, etc."
        )

    with open(filepath) as f:
        return json.load(f)


def check_token_expiry(credentials):
    """Check if the token is still valid. Returns True if valid."""
    expires_at_str = credentials.get("token_expires_at")
    if not expires_at_str:
        return False

    try:
        expires_at = datetime.fromisoformat(expires_at_str)
        now = datetime.now(timezone.utc)
        return now < (expires_at - timedelta(minutes=5))
    except (ValueError, TypeError):
        return False


def refresh_token(credentials):
    """Refresh the OAuth2 access token using client credentials grant."""
    token_url = credentials.get("token_url")
    client_id = credentials.get("client_id")
    client_secret = credentials.get("client_secret")
    scope = credentials.get("scope")

    if not all([token_url, client_id, client_secret]):
        raise ValueError(
            "Missing token refresh credentials (token_url, client_id, client_secret)"
        )

    print("Refreshing OAuth2 token...")
    response = httpx.post(
        token_url,
        data={
            "grant_type": "client_credentials",
            "client_id": client_id,
            "client_secret": client_secret,
            "scope": scope,
        },
        headers={"Content-Type": "application/x-www-form-urlencoded"},
        timeout=30,
    )
    response.raise_for_status()
    token_data = response.json()

    expires_in = token_data.get("expires_in", 3600)
    expires_at = datetime.now(timezone.utc) + timedelta(seconds=expires_in)

    credentials["access_token"] = token_data["access_token"]
    credentials["token_expires_at"] = expires_at.isoformat()

    print(f"Token refreshed. Expires: {expires_at.isoformat()}")
    return credentials


print("Credential functions defined.")

In [ ]:
# Load credentials and refresh token if needed
credentials = load_mcp_credentials()

if not check_token_expiry(credentials):
    credentials = refresh_token(credentials)
else:
    print("Token is still valid.")

gateway_url = credentials["gateway_url"]
access_token = credentials["access_token"]

print(f"Gateway: {gateway_url}")

## 4. Initialize the LLM

Create a connection to Claude via AWS Bedrock using `ChatBedrockConverse`. The agent uses this LLM to reason about queries, decide which MCP tools to call, and formulate Cypher queries against the Neo4j database.

In [ ]:
# Build LLM config - add base_model_id for Claude inference profiles
llm_kwargs = {
    "model": MODEL_ID,
    "region_name": REGION,
    "temperature": 0,
}

if BASE_MODEL_ID:
    llm_kwargs["base_model_id"] = BASE_MODEL_ID

llm = ChatBedrockConverse(**llm_kwargs)

print(f"LLM initialized: {MODEL_ID}")

## 5. Connect to the Neo4j MCP Server

The `MultiServerMCPClient` connects to the Neo4j MCP server through the AgentCore Gateway using Streamable HTTP transport. This gives the agent two tools:

- **get-schema** — Returns the graph database schema (node labels, relationships, properties)
- **execute-query** — Executes read-only Cypher queries against Neo4j

The Gateway handles routing and authentication via the OAuth2 bearer token.

In [ ]:
# Connect to MCP server via AgentCore Gateway
mcp_client = MultiServerMCPClient(
    {
        "neo4j": {
            "transport": "streamable_http",
            "url": gateway_url,
            "headers": {
                "Authorization": f"Bearer {access_token}",
            },
        }
    }
)

# Get available MCP tools as LangChain tools
tools = await mcp_client.get_tools()

print(f"Loaded {len(tools)} tools:")
for tool in tools:
    print(f"  - {tool.name}")

## 6. Create the ReAct Agent

The agent follows the **ReAct pattern** (Reason + Act):
1. Receive a natural language question
2. Reason about what data is needed
3. Call MCP tools (get schema, execute Cypher queries)
4. Observe results and continue reasoning
5. Return a human-readable answer

The system prompt instructs the agent to first retrieve the database schema, use LIMIT on all non-aggregation queries, and format results clearly.

`create_react_agent` from LangGraph builds the same `START → agent → (tools → agent) | END` graph from the [basic_langgraph_agent](basic_langgraph_agent.ipynb) notebook, but with a simpler high-level API.

In [ ]:
SYSTEM_PROMPT = """You are a helpful Neo4j database assistant with access to tools that let you query a Neo4j graph database containing aviation fleet data.

Your capabilities include:
- Retrieve the database schema to understand node labels, relationship types, and properties
- Execute read-only Cypher queries to answer questions about the data
- Do not execute any write Cypher queries

When answering questions about the database:
1. First retrieve the schema to understand the database structure
2. Formulate appropriate Cypher queries based on the actual schema
3. If a query returns no results, explain what you looked for and suggest alternatives
4. Format results in a clear, human-readable way
5. Cite the actual data returned in your response

CRITICAL - Always Use LIMIT:
- For listing/browsing queries: use LIMIT 10 (or LIMIT 25 max)
- For sample data: use LIMIT 5
- For aggregations (COUNT, SUM, AVG): LIMIT is optional
- NEVER return unlimited result sets

Important Cypher notes:
- Use MATCH patterns that align with the actual schema
- For counting, use MATCH (n:Label) RETURN count(n)
- Handle potential NULL values gracefully

Be concise but thorough in your responses."""

# Create the ReAct agent with MCP tools
agent = create_react_agent(llm, tools, prompt=SYSTEM_PROMPT)

print("Agent created successfully!")

## 7. Query the Agent

The aviation database contains Aircraft, Flights, Airports, Operators, Systems, Components, Sensors, Readings, MaintenanceEvents, and Delays.

On the first query, the agent will retrieve the database schema via the `get-schema` MCP tool, then generate and execute Cypher queries to answer questions.

In [ ]:
async def run_agent(question: str):
    """Run the agent with a question and display the response."""
    print(f"Question: {question}")
    print("-" * 60)

    result = await agent.ainvoke({"messages": [HumanMessage(content=question)]})

    final_message = result["messages"][-1]
    print(f"\nResponse:\n{final_message.content}")
    return result

In [ ]:
# Basic Discovery: Database schema overview
result = await run_agent("What is the database schema? Show me all node types and their relationships.")

In [ ]:
# Fleet Operations: Aircraft count and models
result = await run_agent("How many aircraft are in the fleet and what models do we have?")

In [ ]:
# Fleet Operations: List airports
result = await run_agent("List 5 airports with their IATA codes and cities.")

In [ ]:
# Maintenance: Common faults
result = await run_agent("What are the most common maintenance faults across all aircraft?")

In [ ]:
# Flight Operations: Delay analysis
result = await run_agent(
    "What are the most common causes of flight delays and how many minutes do they add on average?"
)

In [ ]:
# Advanced: Cross-domain correlation
result = await run_agent(
    "Which aircraft have both maintenance events AND flight delays? "
    "Show the correlation between maintenance issues and operational delays."
)

## 8. Deploying to AgentCore Runtime

To deploy this agent as a managed service on Amazon Bedrock AgentCore Runtime:

1. **Package** the agent in a Docker container with a `BedrockAgentCoreApp` entrypoint on port 8080
2. **Configure** with `agentcore configure` to generate `.bedrock_agentcore.yaml`
3. **Deploy** with `agentcore deploy` to push the container to ECR and register the runtime
4. **Invoke** the deployed agent programmatically via `boto3.client("bedrock-agentcore").invoke_agent_runtime()`

The deployed agent adds production features:
- **Streaming responses** via Server-Sent Events (SSE)
- **Schema caching** to avoid repeated MCP calls
- **OpenTelemetry tracing** with AWS Distro for OpenTelemetry (ADOT)
- **Auto-scaling** managed by AgentCore Runtime

See the `agentcore-neo4j-mcp-agent/basic-agent/` reference project for the full deployment code.